# ⏩ Flujos de trabajo secuenciales con agentes y modelos de GitHub (Python)

## 📋 Tutorial avanzado de procesamiento secuencial

Este cuaderno demuestra **patrones de flujo de trabajo secuencial** utilizando el Microsoft Agent Framework. Aprenderás a crear tuberías de procesamiento sofisticadas de varios pasos donde los agentes se ejecutan en un orden específico, pasando datos y contexto entre las etapas.

## 🎯 Objetivos de aprendizaje

### 🔄 **Patrones de procesamiento secuencial**
- **Diseño de flujo de trabajo lineal**: Crear tuberías de procesamiento paso a paso
- **Gestión del flujo de datos**: Pasar información entre agentes secuenciales
- **Procesamiento con puertas de etapa**: Implementar puntos de control y etapas de validación
- **Seguimiento del progreso**: Monitorear la ejecución del flujo de trabajo y resultados intermedios

### 🏗️ **Arquitectura de tuberías empresariales**
- **Modelado de procesos de negocio**: Mapear procesos empresariales reales a flujos de trabajo de agentes
- **Aseguramiento de calidad**: Validación y procesos de revisión en múltiples etapas
- **Procesamiento de documentos**: Análisis y transformación secuencial de documentos
- **Producción de contenido**: Flujos editoriales con etapas de revisión y aprobación

### 📊 **Funciones avanzadas de flujo de trabajo**
- **Preservación del contexto**: Mantener el estado a lo largo de las etapas del flujo de trabajo
- **Propagación de errores**: Manejar fallos en el procesamiento secuencial
- **Optimización del rendimiento**: Patrones eficientes de ejecución secuencial
- **Registros de auditoría**: Seguimiento completo de operaciones secuenciales

## ⚙️ Prerrequisitos y configuración

### 📦 **Dependencias**
```bash
pip install agent-framework-core -U
```

### 🔑 **Configuración**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

## 🏢 **Casos de uso de flujo de trabajo secuencial empresarial**

### 📝 **Tubería de procesamiento de documentos**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **Flujo de trabajo de aseguramiento de calidad** 
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **Tubería de producción de contenido**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **Automatización de procesos de negocio**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **Principios de diseño de flujo de trabajo secuencial**

- **🔗 Progresión lineal**: Cada etapa depende de la salida de la etapa anterior
- **📋 Gestión del estado**: Preservar contexto y datos en todas las etapas
- **🛡️ Manejo de errores**: Gestión elegante de fallos en cualquier etapa
- **📊 Monitoreo del progreso**: Rastrear la finalización y rendimiento en cada etapa
- **🔄 Reutilización de etapas**: Diseñar componentes de flujo de trabajo reutilizables

¡Construyamos flujos de trabajo secuenciales sofisticados! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = await provider.create_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = await provider.create_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = await provider.create_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal ChatMessage with an image of a
# living room. The current Message class no longer ships TextContent/DataContent
# helpers, so this migration uses a textual description of the same scene to
# keep the lesson focused on sequential workflow mechanics.
message = Message(
    role="user",
    text=(
        "I am furnishing a modern living room and want pieces that fit a warm, "
        "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
        "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
        "Please find appropriate furniture and give the corresponding price for "
        "each piece, then produce a final purchase quote."
    ),
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
